Celda 1: Carga de Datos Aumentados y Configuración

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, accuracy_score, r2_score

PATH_TRAIN_MENSUAL = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset\train_mensual_aug.xlsx"
PATH_TRAIN_REGISTRO = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset\train_registro_aug.xlsx"
PATH_MODELS_DIR = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\models"

os.makedirs(PATH_MODELS_DIR, exist_ok=True)

df_m = pd.read_excel(PATH_TRAIN_MENSUAL)
df_r = pd.read_excel(PATH_TRAIN_REGISTRO)

print("Datos de entrenamiento cargados. Preparando modelos...")

Datos de entrenamiento cargados. Preparando modelos...


Celda 2: Modelo de Predicción de Demanda (Ventas Mensuales)

In [2]:
features_v = ['monto_mes_anterior', 'servicios_mes_anterior', 'tendencia_3m']
target_v = 'monto_total'

X_v = df_m[features_v]
y_v = df_m[target_v]

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(X_v, y_v, test_size=0.2, random_state=42)

model_ventas = XGBRegressor(
    n_estimators=100, 
    learning_rate=0.05, 
    max_depth=3, 
    random_state=42
)
model_ventas.fit(X_train_v, y_train_v)

pred_v = model_ventas.predict(X_test_v)
mae = mean_absolute_error(y_test_v, pred_v)
r2 = r2_score(y_test_v, pred_v)

print(f"--- MODELO DE DEMANDA (VENTAS) ---")
print(f"Error Medio Absoluto (MAE): S/. {mae:.2f}")
print(f"Puntaje R2 (Ajuste): {r2:.4f}")

joblib.dump(model_ventas, os.path.join(PATH_MODELS_DIR, "model_ventas_mensuales.pkl"))

--- MODELO DE DEMANDA (VENTAS) ---
Error Medio Absoluto (MAE): S/. 288.79
Puntaje R2 (Ajuste): -0.9534


['C:\\Users\\fabri\\OneDrive\\Desktop\\UPAO\\Funeraria_Inventario_Inteligente\\models\\model_ventas_mensuales.pkl']

Celda 3: Modelo de Clasificación de Inventario (Preferencia de Ataúd)

In [3]:
from sklearn.preprocessing import LabelEncoder

le_ataud = LabelEncoder()
df_r['target_ataud'] = le_ataud.fit_transform(df_r['Ataud_Modelo'])

features_a = ['Monto', 'Carroza', 'Auto', 'Cargadores']
X_a = df_r[features_a]
y_a = df_r['target_ataud']

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X_a, y_a, test_size=0.2, random_state=42)

model_inv = RandomForestClassifier(n_estimators=150, max_depth=5, random_state=42)
model_inv.fit(X_train_a, y_train_a)

pred_a = model_inv.predict(X_test_a)
acc = accuracy_score(y_test_a, pred_a)

print(f"--- MODELO DE INVENTARIO (ATAÚDES) ---")
print(f"Precisión del Modelo: {acc * 100:.2f}%")

joblib.dump(model_inv, os.path.join(PATH_MODELS_DIR, "model_inventario_ataud.pkl"))
joblib.dump(le_ataud, os.path.join(PATH_MODELS_DIR, "encoder_ataud.pkl"))

--- MODELO DE INVENTARIO (ATAÚDES) ---
Precisión del Modelo: 33.33%


['C:\\Users\\fabri\\OneDrive\\Desktop\\UPAO\\Funeraria_Inventario_Inteligente\\models\\encoder_ataud.pkl']

Celda 4: Reporte de Importancia de Variables y Predicción Futura

In [4]:
importancia = pd.DataFrame({
    'Variable': features_a,
    'Importancia': model_inv.feature_importances_
}).sort_values(by='Importancia', ascending=False)

print("Factores que más influyen en la elección del ataúd:")
print(importancia)

ultimo_mes_conocido = df_m.iloc[-1:][features_v]
prediccion_futura = model_ventas.predict(ultimo_mes_conocido)[0]

metricas_ia = pd.DataFrame({
    'Modelo': ['Ventas Mensuales (XGBoost)', 'Inventario (Random Forest)'],
    'Métrica Principal': ['MAE (S/.)', 'Accuracy (%)'],
    'Valor': [mae, acc * 100],
    'Predicción Próximo Mes': [prediccion_futura, np.nan]
})

metricas_ia.to_excel(r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset\metricas_ia_final.xlsx", index=False)

print(f"\nPredicción estimada para el próximo mes: S/. {prediccion_futura:.2f}")

Factores que más influyen en la elección del ataúd:
     Variable  Importancia
0       Monto     0.591511
3  Cargadores     0.287324
2        Auto     0.121165
1     Carroza     0.000000

Predicción estimada para el próximo mes: S/. 17656.46
